In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
grades_per_exam = pd.read_csv('./Grades/grades_per_exam.csv',index_col=0)
grades_per_exam

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

In [ ]:
from xgboost import XGBRegressor

In [ ]:
def nota_final_pipeline(w, indicadores):
  theta = pd.read_csv('./Modelos/Cummulative/Configuración al menos un evaluable/theta_one_week{}.csv'.format(w),index_col=0)
  w_ind2 = ind.loc[theta.index]
  w_ind2['theta'] = theta['conf1']

  week_exam = {1:1, 2:1, 3:2, 4:3, 5:4, 6:5, 7:5}
  print('Exam:',week_exam[w])
  quedan_e = pd.read_csv('./Abandono/mastbias_e{}.csv'.format(week_exam[w]),index_col=0)
  if w
  w_ind = w_ind2.loc[quedan_e.index]
  print('Quedan:',len(w_ind))

  w_ind['grade'] = grades_per_exam['Course grade']
  w_ind['grade'] = w_ind['grade'].fillna(0)

  if 'CFA' in w_ind.columns:
      w_ind['CFA'] = w_ind['CFA'].replace('0 problems attempted',0)
  if 'avg_grade' in w_ind.columns:
      w_ind['avg_grade'] = w_ind['avg_grade'].replace('0 problems attempted',0)
  if 'avg_attempts' in w_ind.columns:
      w_ind['avg_attempts'] = w_ind['avg_attempts'].replace('0 problems attempted',0)

  X = w_ind.drop(['grade'],axis=1)
  y = w_ind['grade']

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

  pipe_xgb = Pipeline([('scl', StandardScaler()),
                      ('XGB', XGBRegressor())])

  xgb_param_grid = [{'XGB__learning_rate': [0.1],
                      'XGB__max_depth': [5],
                      'XGB__min_child_weight': [1.0],
                      'XGB__subsample': [1.0],
                      'XGB__n_estimators': [75]}]

  xgb_grid_search = GridSearchCV(estimator=pipe_xgb,
          param_grid=xgb_param_grid,
          scoring='neg_mean_squared_error',
          cv=3)
  grids = [xgb_grid_search]
  names = ['xgb']
  n = 0
  for pipe in grids:
    print(names[n])
    pipe.fit(X_train,y_train)
    n+=1

  mses = {}
  grid_dict = {0: 'XGBoost'}
  for i, model in enumerate(grids):
    mses = -(model.score(X_test,y_test))

  return mses

In [ ]:
rmses = {}
for w in range(1,8):
  print('\nWeek:', w)
  ind =  pd.read_csv('./Modelos/Cummulative/Indicadores/indicadores_week1_{}.csv'.format(w),index_col=0)
  rmse =  nota_final_pipeline(w, indicadores = ind)
  rmses['Week {}'.format(str(w))] = rmse
  print(rmses['Week {}'.format(str(w))])
print(rmses)